# Day 4: Manual Sentiment Labelling

This notebook prepares the labelling file. Fortune must assign every final sentiment label. The assistant must not generate the ground-truth labels.

The frozen dataset contains 111 headlines. The target final labels are `positive`, `neutral`, or `negative`. Use `review` temporarily for an uncertain row, but resolve every `review` row before model training.

## Draft labelling rubric

Rewrite this rubric in your own words in the final notebook before labelling.

- **Positive:** The headline describes falling prices, improved affordability, increased supply, relief, or another clearly favourable development.
- **Negative:** The headline describes rising prices, inflation pressure, scarcity, worsening hardship, or unaffordability.
- **Neutral:** The headline reports information without a clear positive or negative tone.
- **Review:** The headline is ambiguous, mixed, peripheral, or difficult to classify. Resolve it before training.

Label the headline's expressed tone, not whether the event is objectively good or bad. Record a short reason and confidence for every label.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent

CLEAN_DIR = PROJECT_DIR / 'data' / 'clean'
FROZEN_PATH = CLEAN_DIR / 'articles_frozen.csv'
LABEL_PATH = CLEAN_DIR / 'articles_to_label.csv'
LABELED_PATH = CLEAN_DIR / 'articles_labeled.csv'

frozen = pd.read_csv(FROZEN_PATH)
print('Frozen rows:', len(frozen))
print('Frozen columns:', list(frozen.columns))

In [ ]:
if LABEL_PATH.exists():
    labeling = pd.read_csv(LABEL_PATH)
    print('Existing labelling file loaded:', LABEL_PATH)
else:
    labeling = frozen[['article_url', 'headline', 'pub_datetime', 'source_label']].copy()
    labeling.insert(0, 'row_id', range(1, len(labeling) + 1))
    labeling['sentiment_label'] = ''
    labeling['label_reason'] = ''
    labeling['confidence'] = ''
    labeling.to_csv(LABEL_PATH, index=False)
    print('Created labelling file:', LABEL_PATH)

display(labeling[['row_id', 'headline', 'source_label', 'sentiment_label', 'label_reason', 'confidence']].head(15))

## Your labelling task

Open `data/clean/articles_to_label.csv` in a spreadsheet or edit it through Colab. For every row, fill:

- `sentiment_label`: `positive`, `neutral`, or `negative`
- `label_reason`: one short explanation
- `confidence`: `high`, `medium`, or `low`

Do not use VADER, HuggingFace, or another automatic model to create these labels.

In [ ]:
VALID_LABELS = {'positive', 'neutral', 'negative'}
VALID_CONFIDENCE = {'high', 'medium', 'low'}

labels = labeling['sentiment_label'].fillna('').str.strip().str.lower()
confidence = labeling['confidence'].fillna('').str.strip().str.lower()

print('Total rows:', len(labeling))
print('Unlabelled rows:', int(labels.eq('').sum()))
print('Rows needing review:', int(labels.eq('review').sum()))
print('Invalid labels:', int((~labels.isin(VALID_LABELS | {'' , 'review'})).sum()))
print('Label counts:')
display(labels.value_counts(dropna=False).to_frame('count'))
print('Invalid confidence values:', int((~confidence.isin(VALID_CONFIDENCE | {''})).sum()))

In [ ]:
complete = (
    labels.isin(VALID_LABELS).all()
    and confidence.isin(VALID_CONFIDENCE).all()
    and labeling['label_reason'].fillna('').str.strip().ne('').all()
)

if complete:
    labeling.to_csv(LABELED_PATH, index=False)
    print('All labels are complete. Saved:', LABELED_PATH)
else:
    print('Labelling is not complete. Resolve blank, review, invalid, or missing-reason rows first.')

## Exit criteria

Day 4 labelling is complete only when all 111 rows have a final label, a reason, and a confidence value. We will then inspect class balance and begin the VADER and TF-IDF/Logistic Regression comparison.